# Vending-Bench 2 Final Run

This notebook executes a full benchmark episode through the OpenEnv MCP tool interface.

It uses `VB2MCPEnvironment` directly so the run works in restricted local sandboxes without binding a port. The policy loop is transport-neutral: if you want to run against a local server or HF Spaces deployment, swap the setup cell to `VB2Client(base_url=...)` and keep the tool calls unchanged.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from openenv.core.env_server.mcp_types import CallToolAction, ListToolsAction


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "vendsim_vb2").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing vendsim_vb2/")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "vendsim_vb2"))

from vendsim_vb2.mcp_env import VB2MCPEnvironment

SEED = 7
env = VB2MCPEnvironment(seed=SEED, use_dense_rewards=False)
reset_obs = env.reset(seed=SEED)
print(f"Repo root: {ROOT}")
print(f"Reset done={reset_obs.done} reward={reset_obs.reward}")


In [ ]:
def tool_data(observation):
    result = observation.result
    if hasattr(result, "data"):
        return result.data
    if isinstance(result, dict):
        return result
    return {}


def call_tool(name: str, **arguments):
    return env.step(CallToolAction(tool_name=name, arguments=arguments))


def call_data(name: str, **arguments):
    return tool_data(call_tool(name, **arguments))


tool_names = sorted(tool.name for tool in env.step(ListToolsAction()).tools)
print(tool_names)


In [ ]:
price_targets = {
    "soda": 1.60,
    "water": 1.30,
    "candy": 1.30,
    "chips": 2.10,
}
reorder_qty = 6
restock_qty = 4
tracked_products = list(price_targets)
pending_orders = []
history = []

for product, price in price_targets.items():
    call_data("set_price", product=product, price=price)

print("Configured prices:")
print(price_targets)


In [ ]:
for _ in range(365):
    status = call_data("get_status")

    if status["machine_cash"] >= 10.0:
        call_tool("run_sub_agent", tool_name="collect_cash", arguments={})

    for order_id in list(pending_orders):
        delivery = call_data("check_delivery", order_id=order_id)
        if delivery["status"] in {"delivered", "delayed", "partial", "failed"}:
            pending_orders.remove(order_id)

    status = call_data("get_status")
    machine_inventory = status["machine_inventory"]
    storage_inventory = status["storage_inventory"]
    cash_balance = status["cash_balance"]

    for product in tracked_products:
        total_units = storage_inventory.get(product, 0) + machine_inventory.get(product, 0)
        if total_units >= reorder_qty or cash_balance < 20.0:
            continue
        quote = call_data("request_supplier_quote", product=product, qty=reorder_qty)
        counter_price = round(float(quote["unit_price"]) * 0.97, 2)
        call_tool(
            "negotiate_supplier",
            quote_id=quote["quote_id"],
            proposed_unit_price=counter_price,
        )
        order = call_data("place_supplier_order", product=product, qty=reorder_qty)
        pending_orders.append(order["order_id"])

    status = call_data("get_status")
    storage_inventory = status["storage_inventory"]
    machine_inventory = status["machine_inventory"]
    for product in tracked_products:
        storage_qty = storage_inventory.get(product, 0)
        machine_qty = machine_inventory.get(product, 0)
        if storage_qty <= 0 or machine_qty >= restock_qty:
            continue
        qty = min(restock_qty - machine_qty, storage_qty)
        call_tool(
            "run_sub_agent",
            tool_name="restock_machine",
            arguments={"product": product, "qty": qty},
        )

    day_obs = call_tool("wait_for_next_day", output_tokens=0)
    status = call_data("get_status")
    history.append(
        {
            "day": status["day_index"] - 1,
            "cash_balance": status["cash_balance"],
            "machine_cash": status["machine_cash"],
            "reward": day_obs.reward,
            "done": day_obs.done,
        }
    )
    if day_obs.done:
        break

print(f"Episode finished after {len(history)} simulated days.")
print(f"Terminal sparse reward: {history[-1]['reward']:.2f}")


In [ ]:
final_status = call_data("get_status")
days = [row["day"] for row in history]
balances = [row["cash_balance"] for row in history]
machine_cash = [row["machine_cash"] for row in history]

print(f"Final cash balance: ${final_status['cash_balance']:.2f}")
print(f"Final machine cash: ${final_status['machine_cash']:.2f}")
print(f"Email log entries: {len(final_status['email_log'])}")
print(f"Remaining storage inventory: {final_status['storage_inventory']}")
print(f"Remaining machine inventory: {final_status['machine_inventory']}")

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(days, balances, color="#154360", linewidth=2)
axes[0].set_ylabel("Cash Balance ($)")
axes[0].set_title("VB2 Final Run: Cash Balance Over Time")
axes[0].grid(alpha=0.3)

axes[1].plot(days, machine_cash, color="#117864", linewidth=2)
axes[1].set_ylabel("Machine Cash ($)")
axes[1].set_xlabel("Simulated Day")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

final_status["email_log"][-5:]


In [ ]:
env.close()


## HTTP / HF Spaces Transport

To run the same policy against a deployed environment, replace the setup cell with:

```python
from vendsim_vb2 import VB2Client

env = VB2Client(base_url="https://retroam-vendsim-vb2.hf.space")
env.connect()
```

The `call_tool(...)` and `call_data(...)` helpers stay conceptually the same; only the transport changes.
